In [0]:
%python
bronze_path = '/Volumes/workspace/techvenda/filestore/bronze/'
silver_path = '/Volumes/workspace/techvenda/filestore/silver/'
gold_path = '/Volumes/workspace/techvenda/filestore/gold/'
origem_path = '/Volumes/workspace/techvenda/filestore/origem/'

In [0]:
%python
#Tabelas temporarias
bronze_mapeamento= {
    'temp_bronze_clientes' : f'{bronze_path}/clientes/',
    'temp_bronze_itens_pedido' : f'{bronze_path}/itens_pedido/',
    'temp_bronze_pedidos' : f'{bronze_path}/pedidos/',
    'temp_bronze_produtos' : f'{bronze_path}/produtos/',
    'temp_bronze_vendedores' : f'{bronze_path}/vendedores/'

}
for view_name, path in bronze_mapeamento.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)

)

In [0]:
%python

taxa_cancelamento = spark.sql("""
    SELECT 
        COUNT(*) AS total_pedidos,

        SUM(
            CASE 
                WHEN LOWER(TRIM(status_pedido)) = 'cancelado'
                THEN 1
                ELSE 0
            END
        ) AS pedidos_cancelados,

        ROUND(
            (
                SUM(
                    CASE 
                        WHEN LOWER(TRIM(status_pedido)) = 'cancelado'
                        THEN 1
                        ELSE 0
                    END
                ) * 100.0
            ) / COUNT(*),
            2
        
        ) || ' %' AS taxa_cancelamento 

    FROM temp_bronze_pedidos
""")

# Salvar em delta na silver
taxa_cancelamento.write\
    .mode('overwrite')\
        .format('delta')\
            .option('mergeSchema', 'true')\
                .save(f'{gold_path}/percentual_cancelamento')

display(taxa_cancelamento)

In [0]:
%sql
create table if not exists workspace.techvenda.percentual_pedidos_cancelados 
select * from delta. `/Volumes/workspace/techvenda/filestore/gold/percentual_cancelamento/`